# Graficas de medidas con limite de 100 kbit/s

Estas medidas comparan el RTT y la perdida de paquetes obtenidos con `ping` hacia M2 y M3 usando distintos tamanos de paquete. Se contrastan las pruebas sin limitacion de trafico con las pruebas donde `tc` aplica un limite de ancho de banda de 100 kbit/s. Las graficas generadas se guardan en esta misma carpeta.

In [ ]:
from pathlib import Path
import html
import re
from statistics import mean

CARPETA = Path.cwd()


def parse_medida(path):
    texto = path.read_text(encoding='utf-8', errors='replace')
    tam = int(re.search(r'_(\d+)bytes', path.name).group(1))
    condicion = 'Sin limitacion' if 'sin_limitacion' in path.name else '100 kbit/s'
    destino = None
    muestras = {}
    perdida = {}

    for linea in texto.splitlines():
        cabecera = re.match(r'Ping\s+\d+\s+bytes\s+(M\d+)', linea)
        if cabecera:
            destino = cabecera.group(1)
            muestras.setdefault(destino, [])
            continue

        if destino:
            rtt = re.search(r'time=([0-9.]+)\s*ms', linea)
            if rtt:
                muestras[destino].append(float(rtt.group(1)))
                continue

            loss = re.search(r'(\d+(?:\.\d+)?)% packet loss', linea)
            if loss:
                perdida[destino] = float(loss.group(1))

    filas = []
    for host in sorted(set(muestras) | set(perdida)):
        valores = muestras.get(host, [])
        filas.append({
            'archivo': path.name,
            'tamano_bytes': tam,
            'condicion': condicion,
            'destino': host,
            'n_recibidos': len(valores),
            'rtt_medio_ms': mean(valores) if valores else None,
            'rtt_min_ms': min(valores) if valores else None,
            'rtt_max_ms': max(valores) if valores else None,
            'perdida_pct': perdida.get(host, 100.0 if not valores else 0.0),
            'muestras': valores,
        })
    return filas


datos = []
for archivo in sorted(CARPETA.glob('*.txt')):
    datos.extend(parse_medida(archivo))

for fila in datos:
    rtt = 'sin respuesta' if fila['rtt_medio_ms'] is None else f"{fila['rtt_medio_ms']:.3f} ms"
    print(f"{fila['condicion']:15} {fila['tamano_bytes']:4} bytes {fila['destino']}: RTT medio={rtt}, perdida={fila['perdida_pct']:.0f}%")


def color(condicion):
    return '#2878b5' if condicion == 'Sin limitacion' else '#d95f02'


def svg_barras(nombre, titulo, campo, etiqueta_y, maximo=None):
    filas = sorted(datos, key=lambda x: (x['destino'], x['tamano_bytes'], x['condicion']))
    valores = [f[campo] for f in filas if f[campo] is not None]
    if not valores:
        return
    ymax = maximo if maximo is not None else max(valores) * 1.15
    ymax = ymax or 1

    ancho = 980
    alto = 520
    margen_izq = 70
    margen_der = 30
    margen_sup = 60
    margen_inf = 115
    base = alto - margen_inf
    area_h = alto - margen_sup - margen_inf
    paso = (ancho - margen_izq - margen_der) / len(filas)
    barra = paso * 0.68

    partes = [
        f"<svg xmlns='http://www.w3.org/2000/svg' width='{ancho}' height='{alto}' viewBox='0 0 {ancho} {alto}'>",
        "<rect width='100%' height='100%' fill='white'/>",
        f"<text x='{ancho/2}' y='30' text-anchor='middle' font-family='Arial' font-size='22' font-weight='700'>{html.escape(titulo)}</text>",
        f"<line x1='{margen_izq}' y1='{base}' x2='{ancho-margen_der}' y2='{base}' stroke='#333'/>",
        f"<line x1='{margen_izq}' y1='{margen_sup}' x2='{margen_izq}' y2='{base}' stroke='#333'/>",
        f"<text x='20' y='{margen_sup + area_h/2}' transform='rotate(-90 20 {margen_sup + area_h/2})' text-anchor='middle' font-family='Arial' font-size='14'>{html.escape(etiqueta_y)}</text>",
    ]

    for i in range(6):
        v = ymax * i / 5
        y = base - (v / ymax) * area_h
        partes.append(f"<line x1='{margen_izq}' y1='{y:.1f}' x2='{ancho-margen_der}' y2='{y:.1f}' stroke='#ddd'/>")
        partes.append(f"<text x='{margen_izq-10}' y='{y+4:.1f}' text-anchor='end' font-family='Arial' font-size='12'>{v:.2f}</text>")

    for idx, fila in enumerate(filas):
        valor = fila[campo]
        x = margen_izq + idx * paso + (paso - barra) / 2
        h = 0 if valor is None else (valor / ymax) * area_h
        y = base - h
        etiqueta = f"{fila['destino']} {fila['tamano_bytes']}B {fila['condicion']}"
        partes.append(f"<rect x='{x:.1f}' y='{y:.1f}' width='{barra:.1f}' height='{h:.1f}' fill='{color(fila['condicion'])}'/>")
        partes.append(f"<text x='{x + barra/2:.1f}' y='{y-5:.1f}' text-anchor='middle' font-family='Arial' font-size='11'>{'' if valor is None else f'{valor:.2f}'}</text>")
        partes.append(f"<text x='{x + barra/2:.1f}' y='{base+18}' text-anchor='end' transform='rotate(-45 {x + barra/2:.1f} {base+18})' font-family='Arial' font-size='11'>{html.escape(etiqueta)}</text>")

    partes.append("<rect x='720' y='50' width='16' height='16' fill='#2878b5'/><text x='742' y='63' font-family='Arial' font-size='13'>Sin limitacion</text>")
    partes.append("<rect x='720' y='74' width='16' height='16' fill='#d95f02'/><text x='742' y='87' font-family='Arial' font-size='13'>100 kbit/s</text>")
    partes.append('</svg>')
    (CARPETA / nombre).write_text('\n'.join(partes), encoding='utf-8')


svg_barras('rtt_medio_100kbits.svg', 'RTT medio por tamano, destino y condicion', 'rtt_medio_ms', 'RTT medio (ms)')
svg_barras('perdida_paquetes_100kbits.svg', 'Perdida de paquetes por tamano, destino y condicion', 'perdida_pct', 'Perdida (%)', maximo=100)

print('\nGraficas guardadas:')
print(CARPETA / 'rtt_medio_100kbits.svg')
print(CARPETA / 'perdida_paquetes_100kbits.svg')